#**스마트폰 센서 데이터 기반 모션 분류**
# 단계2 : 기본 모델링


## 0.미션

* 데이터 전처리
    * 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리 수행
* 다양한 알고리즘으로 분류 모델 생성
    * 최소 4개 이상의 알고리즘을 적용하여 모델링 수행
    * 성능 비교
    * 각 모델의 성능을 저장하는 별도 데이터 프레임을 만들고 비교
* 옵션 : 다음 사항은 선택사항입니다. 시간이 허용하는 범위 내에서 수행하세요.
    * 상위 N개 변수를 선정하여 모델링 및 성능 비교
        * 모델링에 항상 모든 변수가 필요한 것은 아닙니다.
        * 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교하세요.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것입니다.

## 1.환경설정

In [ ]:
# # 한글 깨짐 방지 위해 구글 코랩에 나눔폰트 설치.
# import sys
# if 'google.colab' in sys.modules:
#     !echo 'debconf debconf/frontend select Noninteractive' | debconf-set-selections
#     !sudo apt-get -qq -y install fonts-nanum
#     import matplotlib.font_manager as fm
#     font_files = fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum'])
#     for fpath in font_files:
#         fm.fontManager.addfont(fpath)

Selecting previously unselected package fonts-nanum.
(Reading database ... 126209 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [ ]:
# # 나눔고딕으로 폰트 변경
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'NanumGothic'

In [ ]:
# # matplot의 폰트와 폰트 크기 확인
# print(plt.rcParams['font.family'], plt.rcParams['font.size'])

['NanumGothic'] 10.0


### (1) 라이브러리 불러오기

* 세부 요구사항
    - 기본적으로 필요한 라이브러리를 import 하도록 코드가 작성되어 있습니다.
    - 필요하다고 판단되는 라이브러리를 추가하세요.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 필요하다고 판단되는 라이브러리를 추가하세요.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# import lightgbm as lgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split ,cross_validate
from sklearn.metrics import *
import joblib

* 함수 생성

In [15]:
# 변수의 특성 중요도 계산하기
def plot_feature_importance(importance, names, result_only = False, topn = 'all'):
    feature_importance = np.array(importance)
    feature_name = np.array(names)

    data={'feature_name':feature_name,'feature_importance':feature_importance}
    fi_temp = pd.DataFrame(data)

    #변수의 특성 중요도 순으로 정렬하기
    fi_temp.sort_values(by=['feature_importance'], ascending=False,inplace=True)
    fi_temp.reset_index(drop=True, inplace = True)

    if topn == 'all' :
        fi_df = fi_temp.copy()
    else :
        fi_df = fi_temp.iloc[:topn]

    #변수의 특성 중요도 그래프로 그리기
    if result_only == False :
        plt.figure(figsize=(10,20))
        sns.barplot(x='feature_importance', y='feature_name', data = fi_df)

        plt.xlabel('importance')
        plt.ylabel('feature name')
        plt.grid()

    return fi_df

### (2) 데이터 불러오기

* 주어진 데이터셋
    * data01_train.csv : 학습 및 검증용
* 세부 요구사항
    - 전체 데이터 'data01_train.csv' 를 불러와 'data' 이름으로 저장합니다.
        - data에서 변수 subject는 삭제합니다.
    - 데이터프레임에 대한 기본 정보를 확인합니다.( .head(), .shape 등)

#### 1) 데이터 로딩

In [16]:
data = pd.read_csv('data01_train.csv')

#### 2) 기본 정보 조회

In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5881 entries, 0 to 5880
Columns: 563 entries, tBodyAcc-mean()-X to Activity
dtypes: float64(561), int64(1), object(1)
memory usage: 25.3+ MB


In [18]:
data.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,21,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,15,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,11,STANDING
3,0.289795,-0.035536,-0.150354,-0.231727,-0.006412,-0.338117,-0.273557,0.014245,-0.347916,0.008288,...,-0.408956,-0.255125,0.612804,0.747381,-0.072944,-0.695819,0.287154,0.111388,17,WALKING
4,0.394807,0.034098,0.091229,0.088489,-0.106636,-0.388502,-0.010469,-0.109680,-0.346372,0.584131,...,-0.563437,-0.044344,-0.845268,-0.974650,-0.887846,-0.705029,0.264952,0.137758,17,WALKING_DOWNSTAIRS


In [19]:
data.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,...,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000,5881.000000
mean,0.274811,-0.017799,-0.109396,-0.603138,-0.509815,-0.604058,-0.628151,-0.525944,-0.605374,-0.465490,...,-0.305883,-0.623548,0.008524,-0.001185,0.009340,-0.007099,-0.491501,0.059299,-0.054594,17.381568
std,0.067614,0.039422,0.058373,0.448807,0.501815,0.417319,0.424345,0.485115,0.413043,0.544995,...,0.322808,0.310371,0.339730,0.447197,0.608190,0.476738,0.509069,0.297340,0.278479,8.938316
min,-0.503823,-0.684893,-1.000000,-1.000000,-0.999844,-0.999667,-1.000000,-0.999419,-1.000000,-1.000000,...,-0.979261,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-0.980143,1.000000
25%,0.262919,-0.024877,-0.121051,-0.992774,-0.977680,-0.980127,-0.993602,-0.977865,-0.980112,-0.936067,...,-0.541969,-0.845985,-0.122361,-0.294369,-0.481718,-0.373345,-0.811397,-0.018203,-0.141555,8.000000
50%,0.277154,-0.017221,-0.108781,-0.943933,-0.844575,-0.856352,-0.948501,-0.849266,-0.849896,-0.878729,...,-0.342923,-0.712677,0.010278,0.005146,0.011448,-0.000847,-0.709441,0.182893,0.003951,19.000000
75%,0.288526,-0.010920,-0.098163,-0.242130,-0.034499,-0.262690,-0.291138,-0.068857,-0.268539,-0.013690,...,-0.127371,-0.501158,0.154985,0.285030,0.499857,0.356236,-0.511330,0.248435,0.111932,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,0.977344,0.478157,1.000000,30.000000


## **2. 데이터 전처리**

* 가변수화, 데이터 분할, NaN 확인 및 조치, 스케일링 등 필요한 전처리를 수행한다.


In [20]:
data.drop('subject', axis=1, inplace=True)

In [21]:
data.head(3)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288508,-0.009196,-0.103362,-0.988986,-0.962797,-0.967422,-0.989000,-0.962596,-0.965650,-0.929747,...,-0.487737,-0.816696,-0.042494,-0.044218,0.307873,0.072790,-0.601120,0.331298,0.165163,STANDING
1,0.265757,-0.016576,-0.098163,-0.989551,-0.994636,-0.987435,-0.990189,-0.993870,-0.987558,-0.937337,...,-0.237820,-0.693515,-0.062899,0.388459,-0.765014,0.771524,0.345205,-0.769186,-0.147944,LAYING
2,0.278709,-0.014511,-0.108717,-0.997720,-0.981088,-0.994008,-0.997934,-0.982187,-0.995017,-0.942584,...,-0.535287,-0.829311,0.000265,-0.525022,-0.891875,0.021528,-0.833564,0.202434,-0.032755,STANDING


In [22]:
sum(data.isna().sum())

0

### (1) 데이터 분할1 : x, y

* 세부 요구사항
    - x, y로 분할합니다.

In [23]:
X = data.drop('Activity', axis=1)
y = data.loc[:, 'Activity']

In [24]:
y.unique()

array(['STANDING', 'LAYING', 'WALKING', 'WALKING_DOWNSTAIRS',
       'WALKING_UPSTAIRS', 'SITTING'], dtype=object)

### (2) 스케일링(필요시)


* 세부 요구사항
    - 스케일링을 필요로 하는 알고리즘 사용을 위해서 코드 수행
    - min-max 방식 혹은 standard 방식 중 한가지 사용.

In [25]:
# 스케일링 필요 없다고 판단

### (3) 데이터분할2 : train, validation

* 세부 요구사항
    - train : val = 8 : 2 혹은 7 : 3
    - random_state 옵션을 사용하여 다른 모델과 비교를 위해 성능이 재현되도록 합니다.

In [26]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

## **3. 기본 모델링**



* 세부 요구사항
    - 최소 4개 이상의 알고리즘을 적용하여 모델링을 수행한다.
    - 각 알고리즘별로 전체 변수로 모델링, 상위 N개 변수를 선택하여 모델링을 수행하고 성능 비교를 한다.
    - (옵션) 알고리즘 중 1~2개에 대해서, 변수 중요도 상위 N개를 선정하여 모델링하고 타 모델과 성능을 비교.
        * 상위 N개를 선택하는 방법은, 변수를 하나씩 늘려가며 모델링 및 성능 검증을 수행하여 적절한 지점을 찾는 것이다.

In [ ]:
acc_df = pd.DataFrame({'모델': [],
                       '논문_정확도(교차검증)': [],
                       '논문_정확도(테스트)': [],
                       '논문_정확도(fi_교차검증)': [],
                       '논문_정확도(fi_테스트)': []
                      })
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)


### (1) 알고리즘1 : RandomForest

In [ ]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train, y_train, cv=5)
print(np.mean(scores['test_score']))

0.9518922011584424


In [ ]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.945042492917847
[[334   0   0   0   0   0]
 [  0 290  20   0   0   0]
 [  0  27 286   0   0   0]
 [  0   0   0 282   4   6]
 [  0   0   0  13 213  13]
 [  0   0   0   4  10 263]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.91      0.94      0.93       310
          STANDING       0.93      0.91      0.92       313
           WALKING       0.94      0.97      0.95       292
WALKING_DOWNSTAIRS       0.94      0.89      0.91       239
  WALKING_UPSTAIRS       0.93      0.95      0.94       277

          accuracy                           0.95      1765
         macro avg       0.94      0.94      0.94      1765
      weighted avg       0.95      0.95      0.94      1765



In [ ]:
joblib.dump(rf_model, 'rf_model_paper.pkl')

['rf_model_paper.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'RandomForest', '논문_정확도(교차검증)': np.mean(scores['test_score']), '논문_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,NaN,NaN


### (2) 알고리즘2 : LogisticRegressor

In [ ]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
scores = cross_validate(lr_model, X_train, y_train, cv=5)
print(np.mean(scores['test_score']))

0.9817778315185975


In [ ]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9875354107648725
[[334   0   0   0   0   0]
 [  0 304   6   0   0   0]
 [  0  14 299   0   0   0]
 [  0   0   0 291   0   1]
 [  0   0   0   0 238   1]
 [  0   0   0   0   0 277]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.96      0.98      0.97       310
          STANDING       0.98      0.96      0.97       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       0.99      1.00      1.00       277

          accuracy                           0.99      1765
         macro avg       0.99      0.99      0.99      1765
      weighted avg       0.99      0.99      0.99      1765



In [ ]:
joblib.dump(lr_model, 'lr_model_paper.pkl')

['lr_model_paper.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'LogisticRegression', '논문_정확도(교차검증)': np.mean(scores['test_score']), '논문_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,NaN,NaN
1,LogisticRegression,0.981778,0.987535,NaN,NaN


### (3) 알고리즘3 : SVM

In [ ]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train, y_train, cv=5)
print(np.mean(scores['test_score']))

0.9800776227158513


In [ ]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9852691218130312
[[334   0   0   0   0   0]
 [  0 304   6   0   0   0]
 [  0  18 295   0   0   0]
 [  0   0   0 291   0   1]
 [  0   0   0   0 239   0]
 [  0   0   0   1   0 276]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.94      0.98      0.96       310
          STANDING       0.98      0.94      0.96       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       1.00      1.00      1.00       277

          accuracy                           0.99      1765
         macro avg       0.99      0.99      0.99      1765
      weighted avg       0.99      0.99      0.99      1765



In [ ]:
joblib.dump(svm_model, 'svm_model_paper.pkl')

['svm_model_paper.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'SVM', '논문_정확도(교차검증)': np.mean(scores['test_score']), '논문_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,NaN,NaN
1,LogisticRegression,0.981778,0.987535,NaN,NaN
2,SVM,0.980078,0.985269,NaN,NaN


### (4) 알고리즘4 : DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train, y_train, cv=5)
print(np.mean(scores['test_score']))

0.8928505703736034


In [ ]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

y_pred = dt_model.predict(X_val)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.8804532577903683
[[334   0   0   0   0   0]
 [  0 284  26   0   0   0]
 [  0  38 275   0   0   0]
 [  0   0   0 265   4  23]
 [  0   0   0  24 186  29]
 [  0   0   0  47  20 210]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.88      0.92      0.90       310
          STANDING       0.91      0.88      0.90       313
           WALKING       0.79      0.91      0.84       292
WALKING_DOWNSTAIRS       0.89      0.78      0.83       239
  WALKING_UPSTAIRS       0.80      0.76      0.78       277

          accuracy                           0.88      1765
         macro avg       0.88      0.87      0.87      1765
      weighted avg       0.88      0.88      0.88      1765



In [ ]:
joblib.dump(dt_model, 'dt_model_paper.pkl')

['dt_model_paper.pkl']

In [ ]:
new_row = pd.DataFrame([{'모델': 'DecisionTreeClassifier', '논문_정확도(교차검증)': np.mean(scores['test_score']), '논문_정확도(테스트)': accuracy_score(y_val, y_pred)}])
acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,NaN,NaN
1,LogisticRegression,0.981778,0.987535,NaN,NaN
2,SVM,0.980078,0.985269,NaN,NaN
3,DecisionTreeClassifier,0.892851,0.880453,NaN,NaN


In [ ]:
acc_df.to_csv('모델 정확도(논문 기준).csv', index=False, encoding='utf-8')

### (5) 알고리즘5 : XGBoost

In [ ]:
# le = LabelEncoder()
# y_xgb = le.fit_transform(y)

In [ ]:
# X_train_xgb, X_val_xgb, y_train_xgb, y_val_xgb = train_test_split(X, y_xgb, test_size=0.3, random_state=42)

In [ ]:
# XGB_model = XGBClassifier(random_state=42)
# XGB_model.fit(X_train_xgb, y_train_xgb)

# y_pred = XGB_model.predict(X_val_xgb)
# y_pred = le.inverse_transform(y_pred)

# y_val = le.inverse_transform(y_val_xgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [ ]:
# new_row = pd.DataFrame([{'model': 'XGBoost', 'accuracy': accuracy_score(y_val, y_pred)}])
# acc_df = pd.concat([acc_df, new_row], ignore_index=True)

In [ ]:
# acc_df

### (5) 알고리즘5 : LightGBM

In [ ]:
# X_lgb = X.copy()
# X_lgb.columns = [f'feature_{i}' for i in range(X.shape[1])]

In [ ]:
# le = LabelEncoder()
# y_lgb = le.fit_transform(y)

In [ ]:
# X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb = train_test_split(X_lgb, y_lgb, test_size=0.3, random_state=42)

In [ ]:
# LGB_model = lgb.LGBMClassifier(random_state=42)
# LGB_model.fit(X_train_lgb, y_train_lgb)  # 경고문이 나옴. 너무 많은 특성 중 쓸모 없는 게 많을 때 이런 경고가 뜰 수 있음. 특성 중요도로 피처 수를 조절할 필요가 있음.

# y_pred = LGB_model.predict(X_val_lgb)
# y_pred = le.inverse_transform(y_pred)

# y_val= le.inverse_transform(y_val_lgb)

# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

In [ ]:
# y_pred = model.predict(x_val)
# y_pred = np.argmax(y_pred, axis=1)

In [ ]:
# y_val = le.inverse_transform(y_val)
# y_pred = le.inverse_transform(y_pred)
# print('accuracy :',accuracy_score(y_val, y_pred))
# print('='*60)
# print(confusion_matrix(y_val, y_pred))
# print('='*60)
# print(classification_report(y_val, y_pred))

# =====특성 중요도(랜덤 포레스트에서 추출)=======================

## 방법1: feature_importance_ 사용

In [ ]:
RFmodel_fi_df = plot_feature_importance(rf_model.feature_importances_, list(X_train), True)
RFmodel_fi_df

,feature_name,feature_importance
0,"angle(X,gravityMean)",0.042171
1,tGravityAcc-max()-X,0.034640
2,tGravityAcc-mean()-X,0.031536
3,tGravityAcc-mean()-Y,0.028904
4,tGravityAcc-min()-X,0.028508
...,...,...
556,fBodyAcc-min()-Z,0.000000
557,fBodyAcc-iqr()-Y,0.000000
558,fBodyAcc-mad()-Z,0.000000
559,"tBodyAcc-arCoeff()-Y,2",0.000000


In [ ]:
# 기존 변수 개수
len(RFmodel_fi_df)

561

In [ ]:
RFmodel_fi_df['cum_importance'] = RFmodel_fi_df['feature_importance'].cumsum() / RFmodel_fi_df['feature_importance'].sum()

In [ ]:
# 누적 중요도 기준 95%까지 필요한 변수 개수
selected_features = RFmodel_fi_df[RFmodel_fi_df['cum_importance'] <= 0.95]['feature_importance']
print(f"누적 중요도 95%까지 선택된 변수 개수: {len(selected_features)}")

누적 중요도 95%까지 선택된 변수 개수: 178


In [ ]:
X_fi = data[RFmodel_fi_df['feature_name'][:300]]
y_fi = data.loc[:, 'Activity']

In [ ]:
X_train_fi, X_val_fi, y_train_fi, y_val_fi = train_test_split(X_fi, y_fi, test_size=0.3, random_state=42)

In [ ]:
# X_test_fi = test_data[RFmodel_fi_df['feature_name'][:300]]

### 랜덤 포레스트

In [ ]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_fi, y_train_fi, cv=5)
print(np.mean(scores['test_score']))

0.9472764218051409


In [ ]:
random_forest_model_fi = RandomForestClassifier(random_state=42)

random_forest_model_fi.fit(X_train_fi, y_train_fi)
y_pred = random_forest_model_fi.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.9756373937677054
[[334   0   0   0   0   0]
 [  0 302   8   0   0   0]
 [  0  14 299   0   0   0]
 [  0   0   0 284   4   4]
 [  0   0   0   3 233   3]
 [  0   0   0   1   6 270]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.96      0.97      0.96       310
          STANDING       0.97      0.96      0.96       313
           WALKING       0.99      0.97      0.98       292
WALKING_DOWNSTAIRS       0.96      0.97      0.97       239
  WALKING_UPSTAIRS       0.97      0.97      0.97       277

          accuracy                           0.98      1765
         macro avg       0.97      0.98      0.98      1765
      weighted avg       0.98      0.98      0.98      1765



In [ ]:
joblib.dump(rf_model, 'rf_model_paper_fi.pkl')

['rf_model_paper_fi.pkl']

In [ ]:
acc_df.loc[0, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[0, '논문_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637
1,LogisticRegression,0.981778,0.987535,NaN,NaN
2,SVM,0.980078,0.985269,NaN,NaN
3,DecisionTreeClassifier,0.892851,0.880453,NaN,NaN


In [ ]:
# from sklearn.feature_selection import SelectKBest, f_classif

# # X: 특성 데이터, y: 타깃 (분류 문제)
# selector = SelectKBest(score_func=f_classif, k=10)  # 상위 10개 피처 선택
# X_selected = selector.fit_transform(X_train, y_train)

In [ ]:
# from sklearn.pipeline import make_pipeline
# from sklearn.feature_selection import SelectKBest, f_classif

# # 파이프라인 생성
# pipeline = make_pipeline(
#     SelectKBest(score_func=f_classif, k=300),
#     RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
# )

# # 교차검증
# scores = cross_validate(pipeline, X_train, y_train, cv=5)
# print("평균 검증 정확도:", np.mean(scores['test_score']))

In [ ]:
# # 2. 특성 선택 (훈련 데이터에만 fit)
# selector = SelectKBest(score_func=f_classif, k=170)
# X_train_selected = selector.fit_transform(X_train, y_train)
# X_test_selected = selector.transform(X_test)

# # 3. 랜덤 포레스트 모델 훈련
# model = RandomForestClassifier(random_state=42)
# model.fit(X_train_selected, y_train)

# # 4. 예측 및 평가
# y_pred = model.predict(X_test_selected)

# print("정확도:", accuracy_score(y_test, y_pred))
# print("분류 리포트:\n", classification_report(y_test, y_pred))

### LinearRegression

In [ ]:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=0.95)
# X_train_pca = pca.fit_transform(X_train)

# print("선택된 주성분 수:", pca.n_components_)
# print("누적 설명력:", pca.explained_variance_ratio_.cumsum())

In [ ]:
# lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
# scores = cross_validate(lr_model, X_train_pca, y_train, cv=5)
# print(np.mean(scores['test_score']))

In [ ]:
# 랜덤 포레스트에서 추출한 중요도 상위 특성 활용
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
scores = cross_validate(lr_model, X_train_fi, y_train_fi, cv=5)
print(np.mean(scores['test_score']))

0.9747313286696787


In [ ]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_fi, y_train_fi)
y_pred = lr_model.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.9807365439093484
[[334   0   0   0   0   0]
 [  0 300  10   0   0   0]
 [  0  20 292   1   0   0]
 [  0   0   0 291   0   1]
 [  0   0   0   0 238   1]
 [  0   0   0   0   1 276]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.94      0.97      0.95       310
          STANDING       0.97      0.93      0.95       313
           WALKING       1.00      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       0.99      1.00      0.99       277

          accuracy                           0.98      1765
         macro avg       0.98      0.98      0.98      1765
      weighted avg       0.98      0.98      0.98      1765



In [ ]:
joblib.dump(lr_model, 'lr_model_paper_fi.pkl')

['lr_model_paper_fi.pkl']

In [ ]:
acc_df.loc[1, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[1, '논문_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737
2,SVM,0.980078,0.985269,NaN,NaN
3,DecisionTreeClassifier,0.892851,0.880453,NaN,NaN


### SVM

In [ ]:
# svm_model = SVC(kernel='linear', C=1.0, random_state=42)
# scores = cross_validate(svm_model, X_train_pca, y_train, cv=5)
# print(np.mean(scores['test_score']))

In [ ]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_fi, y_train_fi, cv=5)
print(np.mean(scores['test_score']))

0.9778905024242353


In [ ]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_fi, y_train_fi)

y_pred = svm_model.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.9801699716713881
[[334   0   0   0   0   0]
 [  0 301   9   0   0   0]
 [  0  24 289   0   0   0]
 [  0   0   0 292   0   0]
 [  0   0   0   0 239   0]
 [  0   0   0   2   0 275]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.93      0.97      0.95       310
          STANDING       0.97      0.92      0.95       313
           WALKING       0.99      1.00      1.00       292
WALKING_DOWNSTAIRS       1.00      1.00      1.00       239
  WALKING_UPSTAIRS       1.00      0.99      1.00       277

          accuracy                           0.98      1765
         macro avg       0.98      0.98      0.98      1765
      weighted avg       0.98      0.98      0.98      1765



In [ ]:
joblib.dump(svm_model, 'svm_model_paper_fi.pkl')

['svm_model_paper_fi.pkl']

In [ ]:
acc_df.loc[2, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[2, '논문_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737
2,SVM,0.980078,0.985269,0.977891,0.980170
3,DecisionTreeClassifier,0.892851,0.880453,NaN,NaN


### DecisionTreeClassifier

In [ ]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_fi, y_train_fi, cv=5)
print(np.mean(scores['test_score']))

0.8938229187556772


In [ ]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_fi, y_train_fi)

y_pred = dt_model.predict(X_val_fi)

print('accuracy :',accuracy_score(y_val_fi, y_pred))
print('='*60)
print(confusion_matrix(y_val_fi, y_pred))
print('='*60)
print(classification_report(y_val_fi, y_pred))

accuracy : 0.8798866855524079
[[334   0   0   0   0   0]
 [  0 283  26   0   0   1]
 [  0  38 275   0   0   0]
 [  0   0   0 265   4  23]
 [  0   0   0  24 186  29]
 [  0   0   0  47  20 210]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.88      0.91      0.90       310
          STANDING       0.91      0.88      0.90       313
           WALKING       0.79      0.91      0.84       292
WALKING_DOWNSTAIRS       0.89      0.78      0.83       239
  WALKING_UPSTAIRS       0.80      0.76      0.78       277

          accuracy                           0.88      1765
         macro avg       0.88      0.87      0.87      1765
      weighted avg       0.88      0.88      0.88      1765



In [ ]:
joblib.dump(dt_model, 'dt_model_paper_fi.pkl')

['dt_model_paper_fi.pkl']

In [ ]:
acc_df.loc[3, '논문_정확도(fi_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[3, '논문_정확도(fi_테스트)'] = accuracy_score(y_val_fi, y_pred)

In [ ]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737
2,SVM,0.980078,0.985269,0.977891,0.980170
3,DecisionTreeClassifier,0.892851,0.880453,0.893823,0.879887


In [ ]:
acc_df.to_csv('모델 정확도(논문 기준_fi).csv', index=False, encoding='utf-8-sig')

##방법2. PCA 활용

In [27]:
acc_df = pd.read_csv('모델 정확도(논문 기준_fi).csv')
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737
2,SVM,0.980078,0.985269,0.977891,0.980170
3,DecisionTreeClassifier,0.892851,0.880453,0.893823,0.879887


In [28]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)

print("선택된 주성분 수:", pca.n_components_)
print("누적 설명력:", pca.explained_variance_ratio_.cumsum()[-1])

선택된 주성분 수: 66
누적 설명력: 0.9502722599485613


In [29]:
X_val_pca = pca.transform(X_val)

### 랜덤포레스트

In [30]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)
scores = cross_validate(rf_model, X_train_pca, y_train, cv=5)
print(np.mean(scores['test_score']))

0.8867784803406906


In [31]:
rf_model = RandomForestClassifier(criterion='gini', n_estimators=90, max_depth=6, random_state=42)

rf_model.fit(X_train_pca, y_train)
y_pred = rf_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.8923512747875354
[[334   0   0   0   0   0]
 [  6 245  59   0   0   0]
 [  0  55 258   0   0   0]
 [  0   0   0 275   4  13]
 [  0   0   0  11 206  22]
 [  0   0   0   5  15 257]]
                    precision    recall  f1-score   support

            LAYING       0.98      1.00      0.99       334
           SITTING       0.82      0.79      0.80       310
          STANDING       0.81      0.82      0.82       313
           WALKING       0.95      0.94      0.94       292
WALKING_DOWNSTAIRS       0.92      0.86      0.89       239
  WALKING_UPSTAIRS       0.88      0.93      0.90       277

          accuracy                           0.89      1765
         macro avg       0.89      0.89      0.89      1765
      weighted avg       0.89      0.89      0.89      1765



In [32]:
acc_df.loc[0, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[0, '논문_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [33]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637,0.886778,0.892351
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737,NaN,NaN
2,SVM,0.980078,0.985269,0.977891,0.980170,NaN,NaN
3,DecisionTreeClassifier,0.892851,0.880453,0.893823,0.879887,NaN,NaN


### 로지스틱회귀

In [34]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', max_iter=5000, random_state=42) # 최적점에 도달 못하여 max_iter=5000로 설정
scores = cross_validate(lr_model, X_train_pca, y_train, cv=5)
print(np.mean(scores['test_score']))

0.962341186046786


In [35]:
lr_model = LogisticRegression(C=2, penalty='l1', solver='liblinear', random_state=42)
lr_model.fit(X_train_pca, y_train)
y_pred = lr_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9711048158640226
[[334   0   0   0   0   0]
 [  3 291  16   0   0   0]
 [  0  20 292   1   0   0]
 [  0   0   0 288   1   3]
 [  0   0   0   0 236   3]
 [  0   0   0   4   0 273]]
                    precision    recall  f1-score   support

            LAYING       0.99      1.00      1.00       334
           SITTING       0.94      0.94      0.94       310
          STANDING       0.95      0.93      0.94       313
           WALKING       0.98      0.99      0.98       292
WALKING_DOWNSTAIRS       1.00      0.99      0.99       239
  WALKING_UPSTAIRS       0.98      0.99      0.98       277

          accuracy                           0.97      1765
         macro avg       0.97      0.97      0.97      1765
      weighted avg       0.97      0.97      0.97      1765



/usr/local/lib/python3.11/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [36]:
acc_df.loc[1, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[1, '논문_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [37]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637,0.886778,0.892351
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737,0.962341,0.971105
2,SVM,0.980078,0.985269,0.977891,0.980170,NaN,NaN
3,DecisionTreeClassifier,0.892851,0.880453,0.893823,0.879887,NaN,NaN


### SVM

In [38]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
scores = cross_validate(svm_model, X_train_pca, y_train, cv=5)
print(np.mean(scores['test_score']))

0.9633135344288595


In [39]:
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_pca, y_train)

y_pred = svm_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.9762039660056657
[[334   0   0   0   0   0]
 [  0 298  12   0   0   0]
 [  0  23 290   0   0   0]
 [  0   0   0 290   1   1]
 [  0   0   0   0 238   1]
 [  0   0   0   3   1 273]]
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       334
           SITTING       0.93      0.96      0.94       310
          STANDING       0.96      0.93      0.94       313
           WALKING       0.99      0.99      0.99       292
WALKING_DOWNSTAIRS       0.99      1.00      0.99       239
  WALKING_UPSTAIRS       0.99      0.99      0.99       277

          accuracy                           0.98      1765
         macro avg       0.98      0.98      0.98      1765
      weighted avg       0.98      0.98      0.98      1765



In [40]:
acc_df.loc[2, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[2, '논문_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [41]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637,0.886778,0.892351
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737,0.962341,0.971105
2,SVM,0.980078,0.985269,0.977891,0.980170,0.963314,0.976204
3,DecisionTreeClassifier,0.892851,0.880453,0.893823,0.879887,NaN,NaN


### 결정 트리

In [42]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
scores = cross_validate(dt_model, X_train_pca, y_train, cv=5)
print(np.mean(scores['test_score']))

0.7857126425934009


In [43]:
dt_model = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_model.fit(X_train_pca, y_train)

y_pred = dt_model.predict(X_val_pca)

print('accuracy :',accuracy_score(y_val, y_pred))
print('='*60)
print(confusion_matrix(y_val, y_pred))
print('='*60)
print(classification_report(y_val, y_pred))

accuracy : 0.8062322946175637
[[330   3   1   0   0   0]
 [  5 240  65   0   0   0]
 [  0  63 250   0   0   0]
 [  0   0   0 260  24   8]
 [  0   0   0  58 165  16]
 [  0   0   0  72  27 178]]
                    precision    recall  f1-score   support

            LAYING       0.99      0.99      0.99       334
           SITTING       0.78      0.77      0.78       310
          STANDING       0.79      0.80      0.79       313
           WALKING       0.67      0.89      0.76       292
WALKING_DOWNSTAIRS       0.76      0.69      0.73       239
  WALKING_UPSTAIRS       0.88      0.64      0.74       277

          accuracy                           0.81      1765
         macro avg       0.81      0.80      0.80      1765
      weighted avg       0.82      0.81      0.81      1765



In [44]:
acc_df.loc[3, '논문_정확도(pca_교차검증)'] = np.mean(scores['test_score'])
acc_df.loc[3, '논문_정확도(pca_테스트)'] = accuracy_score(y_val, y_pred)

In [45]:
acc_df

,모델,논문_정확도(교차검증),논문_정확도(테스트),논문_정확도(fi_교차검증),논문_정확도(fi_테스트),논문_정확도(pca_교차검증),논문_정확도(pca_테스트)
0,RandomForest,0.951892,0.945042,0.947276,0.975637,0.886778,0.892351
1,LogisticRegression,0.981778,0.987535,0.974731,0.980737,0.962341,0.971105
2,SVM,0.980078,0.985269,0.977891,0.980170,0.963314,0.976204
3,DecisionTreeClassifier,0.892851,0.880453,0.893823,0.879887,0.785713,0.806232


In [46]:
acc_df.to_csv('모델 정확도(논문 기준_fi_pca).csv', index=False, encoding='utf-8-sig')